# CardioGuard Phase 2: personalized rhythm-anomaly model (MIT-BIH)

**Question:** does scoring each new window against *the user's own baseline* flag arrhythmia episodes better than a generic 60-100 bpm rule or a one-size-fits-all population model?

**Setup**
- Input signal: beat times only (RR intervals), the same thing a watch gives the phone. No ECG morphology.
- Windows: 30 s. Window label = 1 if it holds >= 2 ectopic beats (cardiologist annotations, used for *labels only*).
- Each record plays one "user": first 10 min = calibration baseline, the rest is scored.
- Leave-one-record-out: the model never sees the held-out person's labels.

Set env var `CG_SOURCE=synthetic` to run offline on fake data (pipeline test only, not results).

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from cardioguard_ml.data import load_mitdb, synthetic_records
from cardioguard_ml.features import window_features
from cardioguard_ml.model import (leave_one_record_out, summarize, sustained_alerts, DETECTORS)

SOURCE = os.environ.get("CG_SOURCE", "mitdb")
N_BASE = 20   # 20 windows x 30 s = 10 min calibration
print("source:", SOURCE)

In [ ]:
raw = load_mitdb("data/mitdb") if SOURCE == "mitdb" else synthetic_records(20, 30)
records = {k: window_features(t, s) for k, (t, s) in raw.items()}
records = {k: v for k, v in records.items() if len(v) > N_BASE + 5}
tot = pd.concat(records.values())
print(f"{len(records)} records, {len(tot)} windows, {tot['label'].mean():.1%} anomalous")

## Leave-one-record-out evaluation

In [ ]:
results = leave_one_record_out(records, n_base=N_BASE)
pooled, per_record = summarize(results)
pooled.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for d in DETECTORS:
    fpr, tpr, _ = roc_curve(results["label"], results[d])
    ax.plot(fpr, tpr, label=f"{d} (AUC {pooled.loc[d,'AUROC']:.2f})")
ax.plot([0,1],[0,1],"k:", lw=1); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Pooled ROC, held-out records"); ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

### Per-record AUROC (does it work for everyone, or just on average?)

In [ ]:
per_record.round(2).sort_values("personalized_if")

In [ ]:
wins = (per_record["personalized_if"] > per_record["fixed_range"]).mean()
print(f"personalized_if beats fixed_range on {wins:.0%} of records; "
      f"beats population_if on {(per_record['personalized_if'] > per_record['population_if']).mean():.0%}")

## Two-tier alerting
"Worth noting" = any window over a threshold (in-app only). "Urgent" = **2+ consecutive** windows over the threshold. Threshold here = 99th percentile of normal-window scores, chosen on the evaluation data, so treat these numbers as optimistic until validated on a separate split.

In [ ]:
rows = []
for d in DETECTORS:
    thr = results.loc[results.label == 0, d].quantile(0.99)
    rows.append({"detector": d, **sustained_alerts(results, d, thr, k=2)})
pd.DataFrame(rows).set_index("detector").round(3)

## Caveats to state in the writeup
- Labels are ectopic-beat counts from annotations, not "cardiac events"; this is rhythm-anomaly flagging, not diagnosis.
- Only 44-48 people, 30 min each, mostly clinical patients. Baselines are short (10 min), not the 1-2 weeks the app will use.
- Thresholds/tiering tuned on evaluation data; report a held-out split before quoting numbers.
- MIT-BIH beat times come from annotations; a wrist device will have noisier beat detection (motion artifact).